### MoM Site Visit vs Sales Closure

Excel mirror of **Site Visit vs Sales Closure** plus flowchart extras:

| Sheet | Content |
|-------|---------|
| `RI_NRI` | RI/NRI × month × source → SV count, Sales closure count |
| `Pincode` | Pincode × month × source |
| `MicroMarket` | Micromarket × month × source |
| `NRI_Country_State` | NRI MoM by Country / State |
| `NRI_Organic_Paid` | NRI MoM Organic vs Paid (`Platform=Google` + `CID=NA` → Organic) |
| `Pincode_Top30` | Top 30 SV pincodes, Top 30 closure pincodes, overlap / anomaly |

**Definitions**
- Site visit = earliest **Post Site Visit** in window
- Month = that PSV month
- Sales closure = same lead later reached **Sales Closure** (counted on PSV month)
- Source = `leadsources.name` via `leads.source`
- RI/NRI = `country` India → RI, else NRI (blank → Unknown)
- Micromarket = `visitorlogs.micromarket` (leads.micromarket is empty)
- Pincode = `postal_code` from `zipcodeaddress` JSON (lead, else visitorlog)


In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
from typing import Any

import awswrangler as wr
import boto3
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows

ANALYSIS_START_DATE = date(2026, 1, 1)
ANALYSIS_END_DATE = date.today()

SV_STAGE = "Post Site Visit"
CLOSURE_STAGE = "Sales Closure"
GOOGLE_PLATFORM_ID = "63d7a214e191001f59dd454f"  # googleadsreportmetas / Brand Phrase placements

OUTPUT_XLSX = Path("/Users/lekhanigam/data/leads_sv_closure_reports/Site_Visit_vs_Sales_Closure.xlsx")
ENV_CANDIDATES = [Path("/Users/lekhanigam/data/.env"), Path(".env")]

START_TS = f"TIMESTAMP '{ANALYSIS_START_DATE.isoformat()} 00:00:00'"
END_EXCL = ANALYSIS_END_DATE + timedelta(days=1)
END_TS_EXCL = f"TIMESTAMP '{END_EXCL.isoformat()} 00:00:00'"

print("Window:", ANALYSIS_START_DATE, "→", ANALYSIS_END_DATE)
print("Output:", OUTPUT_XLSX)


### Athena Connection


In [ ]:
def _aws_session(env: dict) -> "boto3.Session":
    """SSO profile — run: aws sso login --profile <AWS_PROFILE>"""
    profile = (env.get("AWS_PROFILE") or "asbl").strip()
    region = (env.get("AWS_REGION") or "ap-south-1").strip()
    session = boto3.Session(profile_name=profile, region_name=region)
    try:
        session.client("sts").get_caller_identity()
    except Exception as e:
        raise RuntimeError(
            f"AWS SSO login required. Run:\n  aws sso login --profile {profile}\n({e})"
        ) from e
    return session


def _load_env(path: Path) -> dict[str, str]:
    out: dict[str, str] = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip()
    return out


_env: dict[str, str] = {}
for _p in ENV_CANDIDATES:
    if _p.exists():
        _env = _load_env(_p)
        print("Loaded", _p)
        break
else:
    raise FileNotFoundError("No .env found")

AWS_REGION = _env.get("AWS_REGION", "ap-south-1")
ATHENA_DATABASE = _env.get("ATHENA_DATABASE", "leadsdb-prod")
ATHENA_WORKGROUP = _env.get("ATHENA_WORKGROUP", "primary")
ATHENA_S3_OUTPUT = _env.get("ATHENA_OUTPUT_LOCATION", "s3://metabase-asbl/athena-query-results/")

AWS_PROFILE = _env.get("AWS_PROFILE", "asbl")
session = _aws_session(_env)
print("AWS profile:", AWS_PROFILE)
print("AWS account:", session.client("sts").get_caller_identity().get("Account"))


def athena_query(sql: str) -> pd.DataFrame:
    return wr.athena.read_sql_query(
        sql=sql,
        database=ATHENA_DATABASE,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_S3_OUTPUT,
        ctas_approach=False,
    )


probe = athena_query("SELECT 1 AS ok FROM leads LIMIT 1")
assert not probe.empty, "Cannot read leads — check credentials / workgroup / database"
print("Athena OK |", ATHENA_DATABASE, "|", ATHENA_WORKGROUP)


### Build base grain in Athena

1 row per lead with a PSV in the window, enriched with source / geo / organic flag.
Aggregations stay in Athena where possible.


In [ ]:
sv_sql = SV_STAGE.replace("'", "''")
cl_sql = CLOSURE_STAGE.replace("'", "''")
gplat = GOOGLE_PLATFORM_ID.replace("'", "''")

# Lead-level grain: earliest PSV in window + dims + closure flag
LEAD_SQL = f"""
WITH psv AS (
  SELECT
    CAST(h.lead AS varchar) AS lead_oid,
    MIN(h.createdat) AS psv_at
  FROM leadstagehistories h
  WHERE CAST(h.stage AS varchar) = '{sv_sql}'
    AND h.createdat >= {START_TS}
    AND h.createdat < {END_TS_EXCL}
  GROUP BY 1
),
closure AS (
  SELECT
    CAST(h.lead AS varchar) AS lead_oid,
    MIN(h.createdat) AS closed_at
  FROM leadstagehistories h
  WHERE CAST(h.stage AS varchar) = '{cl_sql}'
  GROUP BY 1
),
ct_one AS (
  SELECT
    CAST(cid AS varchar) AS cid,
    MAX(CAST(placement AS varchar)) AS placement_fk,
    MAX(CAST(source AS varchar)) AS utm_source_fk
  FROM campaigntrackers
  WHERE cid IS NOT NULL AND CAST(cid AS varchar) NOT IN ('', 'NA', 'N/A', 'null')
  GROUP BY 1
),
vl_one AS (
  SELECT
    CAST(lead AS varchar) AS lead_oid,
    MAX(NULLIF(TRIM(CAST(micromarket AS varchar)), '')) AS micromarket,
    MAX(
      NULLIF(
        regexp_extract(CAST(zipcodeaddress AS varchar), 'postal_code"\\s*:\\s*"([0-9A-Za-z-]{{3,12}})"', 1),
        ''
      )
    ) AS pin_vl,
    MAX(LOWER(NULLIF(TRIM(CAST(sourcecampaign_utm_source AS varchar)), ''))) AS utm_source
  FROM visitorlogs
  WHERE lead IS NOT NULL
  GROUP BY 1
)
SELECT
  date_format(p.psv_at, '%Y-%m') AS month_key,
  date_format(p.psv_at, '%M') AS month_name,
  CASE
    WHEN LOWER(TRIM(CAST(l.country AS varchar))) IN ('india', 'in') THEN 'RI'
    WHEN l.country IS NULL OR TRIM(CAST(l.country AS varchar)) IN ('', 'null', 'None', 'N/A', 'NA') THEN 'Unknown'
    ELSE 'NRI'
  END AS ri_nri,
  COALESCE(
    NULLIF(
      regexp_extract(CAST(l.zipcodeaddress AS varchar), 'postal_code"\\s*:\\s*"([0-9A-Za-z-]{{3,12}})"', 1),
      ''
    ),
    v.pin_vl,
    'Unknown'
  ) AS pincode,
  COALESCE(v.micromarket, 'Unknown') AS micromarket,
  COALESCE(NULLIF(TRIM(CAST(ls.name AS varchar)), ''), 'Unknown') AS source,
  COALESCE(NULLIF(TRIM(CAST(l.country AS varchar)), ''), 'Unknown') AS country,
  COALESCE(NULLIF(TRIM(CAST(l.state AS varchar)), ''), 'Unknown') AS state,
  CASE
    WHEN (
      l.utm_campaign IS NULL
      OR TRIM(CAST(l.utm_campaign AS varchar)) IN ('', 'NA', 'N/A', 'null', 'None')
    )
    AND (
      CAST(pl.platform AS varchar) = '{gplat}'
      OR LOWER(TRIM(CAST(us.name AS varchar))) IN ('google', 'google+', 'googleads', 'ga')
      OR COALESCE(v.utm_source, '') IN ('google', 'organic', 'google_organic')
      OR COALESCE(v.utm_source, '') LIKE 'google%'
    )
    THEN 'Organic'
    ELSE 'Paid'
  END AS organic_paid,
  p.lead_oid,
  CASE
    WHEN c.closed_at IS NOT NULL AND c.closed_at >= p.psv_at THEN 1
    ELSE 0
  END AS is_closed
FROM psv p
JOIN leads l ON CAST(l._id AS varchar) = p.lead_oid
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar) = CAST(l.source AS varchar)
LEFT JOIN ct_one ct ON ct.cid = TRIM(CAST(l.utm_campaign AS varchar))
LEFT JOIN placements pl ON CAST(pl._id AS varchar) = ct.placement_fk
LEFT JOIN utmsources us ON CAST(us._id AS varchar) = ct.utm_source_fk
LEFT JOIN vl_one v ON v.lead_oid = p.lead_oid
LEFT JOIN closure c ON c.lead_oid = p.lead_oid
"""

print("Lead-level SQL ready")


### Aggregate sheets

One Athena lead-level pull, then groupby → RI_NRI / Pincode / MicroMarket / NRI extras.


In [ ]:
print("Pulling lead-level grain from Athena (one pass)…")
leads_base = athena_query(LEAD_SQL)
print("  leads:", len(leads_base),
      "| SV months:", leads_base["month_key"].nunique(),
      "| closures:", int(leads_base["is_closed"].sum()))
assert not leads_base.empty, "No Post Site Visit leads in window"

for c in ["ri_nri", "pincode", "micromarket", "source", "country", "state", "organic_paid", "month_name"]:
    leads_base[c] = leads_base[c].fillna("Unknown").astype(str).replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
leads_base["is_closed"] = pd.to_numeric(leads_base["is_closed"], errors="coerce").fillna(0).astype(int)


def pivot_dim(df: pd.DataFrame, dim: str, dim_label: str) -> pd.DataFrame:
    g = (
        df.groupby([dim, "month_name", "month_key", "source"], as_index=False)
        .agg(**{
            "Lead site visit count": ("lead_oid", "count"),
            "Sales closure count": ("is_closed", "sum"),
        })
        .rename(columns={dim: dim_label})
        .sort_values(["month_key", dim_label, "source"])
        .reset_index(drop=True)
    )
    return g


ri = pivot_dim(leads_base, "ri_nri", "RI/NRI")
pin = pivot_dim(leads_base, "pincode", "Pincode")
mm = pivot_dim(leads_base, "micromarket", "MicroMarket")

nri = leads_base[leads_base["ri_nri"] == "NRI"].copy()
nri_geo = (
    nri.groupby(["country", "state", "month_name", "month_key"], as_index=False)
    .agg(**{
        "Lead site visit count": ("lead_oid", "count"),
        "Sales closure count": ("is_closed", "sum"),
    })
    .sort_values(["month_key", "country", "state"])
    .reset_index(drop=True)
)
nri_op = (
    nri.groupby(["organic_paid", "month_name", "month_key", "source"], as_index=False)
    .agg(**{
        "Lead site visit count": ("lead_oid", "count"),
        "Sales closure count": ("is_closed", "sum"),
    })
    .rename(columns={"organic_paid": "Organic/Paid"})
    .sort_values(["month_key", "Organic/Paid", "source"])
    .reset_index(drop=True)
)

print("RI_NRI", len(ri), "| Pincode", len(pin), "| MicroMarket", len(mm))
print("NRI geo", len(nri_geo), "| NRI organic/paid", len(nri_op))
display(ri.head(12))
display(mm.head(8))


### Pincode Top 30 + overlap / anomaly


In [ ]:
pin_tot = (
    pin.groupby("Pincode", as_index=False)
    .agg(**{
        "Lead site visit count": ("Lead site visit count", "sum"),
        "Sales closure count": ("Sales closure count", "sum"),
    })
)
pin_tot = pin_tot[pin_tot["Pincode"].astype(str).str.lower() != "unknown"].copy()

top_sv = pin_tot.nlargest(30, "Lead site visit count").reset_index(drop=True)
top_sv.insert(0, "rank", top_sv.index + 1)
top_sv["list"] = "Top30_SiteVisit"

top_cl = pin_tot.nlargest(30, "Sales closure count").reset_index(drop=True)
top_cl.insert(0, "rank", top_cl.index + 1)
top_cl["list"] = "Top30_SalesClosure"

sv_set = set(top_sv["Pincode"].astype(str))
cl_set = set(top_cl["Pincode"].astype(str))
overlap = sorted(sv_set & cl_set)
sv_only = sorted(sv_set - cl_set)
cl_only = sorted(cl_set - sv_set)

flags = pd.DataFrame(
    [{"Pincode": p, "flag": "overlap_high_SV_and_closure"} for p in overlap]
    + [{"Pincode": p, "flag": "anomaly_high_SV_low_closure"} for p in sv_only]
    + [{"Pincode": p, "flag": "anomaly_high_closure_not_top_SV"} for p in cl_only]
)

# attach totals for flagged pins
flags = flags.merge(pin_tot, on="Pincode", how="left")
pin_top30 = pd.concat(
    [
        top_sv.assign(section="A_Top30_SiteVisit"),
        top_cl.assign(section="B_Top30_SalesClosure"),
        flags.assign(section="C_Overlap_Anomaly", rank=np.nan, list=flags["flag"]),
    ],
    ignore_index=True,
    sort=False,
)

print("Overlap:", len(overlap), "| SV-only anomaly:", len(sv_only), "| Closure-only anomaly:", len(cl_only))
display(flags.head(20))


### Validate + Export Excel


In [ ]:
def _sheet_frame(df: pd.DataFrame, drop_cols: list[str] | None = None) -> pd.DataFrame:
    out = df.copy()
    if drop_cols:
        out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors="ignore")
    for c in out.columns:
        if "count" in str(c).lower():
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)
    return out


sheets = {
    "RI_NRI": _sheet_frame(ri, ["month_key"]),
    "Pincode": _sheet_frame(pin, ["month_key"]),
    "MicroMarket": _sheet_frame(mm, ["month_key"]),
    "NRI_Country_State": _sheet_frame(nri_geo, ["month_key"]),
    "NRI_Organic_Paid": _sheet_frame(nri_op, ["month_key"]),
    "Pincode_Top30": _sheet_frame(pin_top30),
}

# totals must match across the 3 primary dims
sv_ri = int(sheets["RI_NRI"]["Lead site visit count"].sum())
sv_pin = int(sheets["Pincode"]["Lead site visit count"].sum())
sv_mm = int(sheets["MicroMarket"]["Lead site visit count"].sum())
cl_ri = int(sheets["RI_NRI"]["Sales closure count"].sum())
assert sv_ri == sv_pin == sv_mm, (sv_ri, sv_pin, sv_mm)
assert cl_ri == int(sheets["Pincode"]["Sales closure count"].sum()) == int(sheets["MicroMarket"]["Sales closure count"].sum())
print("Validation PASS | total SV", sv_ri, "| total closures", cl_ri)

HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(color="FFFFFF", bold=True)
THIN = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

wb = Workbook()
wb.remove(wb.active)

for name, df in sheets.items():
    ws = wb.create_sheet(name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=1):
        for c_idx, val in enumerate(row, start=1):
            cell = ws.cell(r_idx, c_idx, val)
            cell.border = THIN
            cell.alignment = Alignment(vertical="center")
            if r_idx == 1:
                cell.fill = HEADER_FILL
                cell.font = HEADER_FONT
    for col in ws.columns:
        letter = get_column_letter(col[0].column)
        width = min(40, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col) + 2))
        ws.column_dimensions[letter].width = width
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = "A2"

wb.save(OUTPUT_XLSX)
print("Wrote", OUTPUT_XLSX)
for name, df in sheets.items():
    print(f"  {name}: {len(df)} rows")


### Final check


In [ ]:
print("Analysis Period:", ANALYSIS_START_DATE, "to", ANALYSIS_END_DATE)
print("SV stage:", SV_STAGE, "| Closure stage:", CLOSURE_STAGE)
print("Excel:", OUTPUT_XLSX)
print("Sheets:", list(sheets))
print("RI/NRI mix:")
display(
    sheets["RI_NRI"]
    .groupby("RI/NRI", as_index=False)[["Lead site visit count", "Sales closure count"]]
    .sum()
)
print("Organic/Paid (NRI):")
display(
    sheets["NRI_Organic_Paid"]
    .groupby("Organic/Paid", as_index=False)[["Lead site visit count", "Sales closure count"]]
    .sum()
)
